# 02b · Paper 2 extra seeds (1 and 2) for the six G/P models
**Runtime: T4 GPU.** Run all. Same code, data, folds and settings as notebook 02 (v3, 200 epochs); only the seed changes.
Writes to the same folder `results/p2_cbis_runs_e200` as `<model>_fold<k>_seed1/2`; seed-0 runs are never touched.
Resumable: finished runs are skipped, so after a disconnect just Run all again. Order: all of seed 1 first, then seed 2.
60 runs, about 25 min each (~26 GPU hours, ~30 units). The last cell releases the GPU when training ends.

In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, importlib

PROJECT  = '/content/drive/MyDrive/quantum-placement'
DATA_DIR = f'{PROJECT}/data/processed/cbis_mass_256'
CODE_DIR = f'{PROJECT}/code'
RUNS_DIR = f'{PROJECT}/results/p2_cbis_runs_e200'     # v3: 200-epoch runs (old 50-epoch runs stay in p2_cbis_runs)
TIME_DIR = f'{PROJECT}/results/_timing'
QSEG_DIR = '/content/drive/MyDrive/QSEG_FINAL/code'            # Paper 1 package (folder that contains qseg/)
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'Paper 1 qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for d in (CODE_DIR, RUNS_DIR, TIME_DIR): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{DATA_DIR}/folds.csv'), 'Run notebook 01 first'
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
print('qseg :', QSEG_DIR); print('data :', DATA_DIR); print('runs :', RUNS_DIR)


In [ ]:
# 2 · Use the Paper 2 library already on Drive (written by notebook 02; not rewritten here)
import importlib, qseg2; importlib.reload(qseg2)
import tensorflow as tf, numpy as np, pandas as pd
print('qseg2 from:', qseg2.__file__)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 GPU'


In [ ]:
# 3 · Data + shared config (Paper 1 settings)
from qseg.config import Config
cfg = Config(work_dir=RUNS_DIR, max_epochs=200)   # v3: up to 200 epochs; batch 16, Adam 1e-4, early stop patience 10 after epoch 15
X, Y, DF = qseg2.load_cbis(DATA_DIR)
print(X.shape, Y.shape, 'ROIs:', len(DF), '| patients:', DF.patient.nunique())
print(DF.groupby('fold').agg(rois=('roi_id','size'), patients=('patient','nunique')).T.to_string())


In [ ]:
# 4 · Training: seeds 1 and 2, six G/P models, 5 folds (resumable)
MODELS_TO_RUN = ['g_q', 'g_cl', 'g_id', 'p_q', 'p_cl', 'p_id']
FOLDS, SEEDS  = [0, 1, 2, 3, 4], [1, 2]
DATA = (X, Y, DF)
def n_done():
    return sum(os.path.exists(f'{RUNS_DIR}/{m}_fold{f}_seed{s}/summary.json') for s in SEEDS for f in FOLDS for m in MODELS_TO_RUN)
import time
print(f'already finished: {n_done()} / {len(SEEDS) * len(FOLDS) * len(MODELS_TO_RUN)}')
for seed in SEEDS:
    for fold in FOLDS:
        for name in MODELS_TO_RUN:
            qseg2.run_fold(cfg, DATA, name, fold, seed, out_root=RUNS_DIR, min_epochs=15, verbose=0)
            print(time.strftime('%H:%M UTC', time.gmtime()), f'| done {n_done()} / 60', flush=True)
print('all requested runs finished')


In [ ]:
# 5 · Auto-shutdown: save Drive, then release the GPU (runs only after cell 4 finishes)
import time
from google.colab import drive, runtime
print('training cell finished at', time.strftime('%H:%M:%S UTC', time.gmtime()))
drive.flush_and_unmount()
print('Drive flushed')
time.sleep(30)
runtime.unassign()
